## Resultants in Sympy

The *resultant* of two polynomials is the determinant of their Sylvester matrix and is used to compute the intersection of two algebraic curves (Meliot, 2020).

(Semaev, 2004) proves that higher term summation polynomials can be constructed recursively using resultants.

We demonstrate how to construct the fourth summation polynomial from the third.

In [2]:
import sympy as sp
from sympy.polys.polytools import resultant

x1, x2, x3, x4, x = sp.symbols('x1 x2 x3 x4 x')
a, b = sp.symbols('a b')
# Semaev F3
def F3(x1, x2, x3):
    return (
        (x1 - x2)**2 * x**2
        - 2*((x1 + x2)*(x1*x2 + a) + 2*b)*x3
        + (x1*x2 - a)**2
        - 4*b*(x1 + x2)
    )
#Construct F3 sharing intermediate x
f_left  = F3(x1, x2, x)
f_right = F3(x3, x4, x)
#Find the resultant
F4 = resultant(f_left, f_right, x)
F4_reduced = sp.factor(F4)

# Optional: structure of factorization
F4_structure = sp.factor_list(F4)

print(F4_reduced)
print(F4_structure)


a**4*x1**4 - 4*a**4*x1**3*x2 - 4*a**4*x1**3*x3 - 4*a**4*x1**3*x4 + 6*a**4*x1**2*x2**2 + 4*a**4*x1**2*x2*x3 + 4*a**4*x1**2*x2*x4 + 6*a**4*x1**2*x3**2 + 4*a**4*x1**2*x3*x4 + 6*a**4*x1**2*x4**2 - 4*a**4*x1*x2**3 + 4*a**4*x1*x2**2*x3 + 4*a**4*x1*x2**2*x4 + 4*a**4*x1*x2*x3**2 - 40*a**4*x1*x2*x3*x4 + 4*a**4*x1*x2*x4**2 - 4*a**4*x1*x3**3 + 4*a**4*x1*x3**2*x4 + 4*a**4*x1*x3*x4**2 - 4*a**4*x1*x4**3 + a**4*x2**4 - 4*a**4*x2**3*x3 - 4*a**4*x2**3*x4 + 6*a**4*x2**2*x3**2 + 4*a**4*x2**2*x3*x4 + 6*a**4*x2**2*x4**2 - 4*a**4*x2*x3**3 + 4*a**4*x2*x3**2*x4 + 4*a**4*x2*x3*x4**2 - 4*a**4*x2*x4**3 + a**4*x3**4 - 4*a**4*x3**3*x4 + 6*a**4*x3**2*x4**2 - 4*a**4*x3*x4**3 + a**4*x4**4 - 8*a**3*b*x1**3 + 8*a**3*b*x1**2*x2 + 8*a**3*b*x1**2*x3 + 8*a**3*b*x1**2*x4 + 8*a**3*b*x1*x2**2 - 16*a**3*b*x1*x2*x3 - 16*a**3*b*x1*x2*x4 + 8*a**3*b*x1*x3**2 - 16*a**3*b*x1*x3*x4 + 8*a**3*b*x1*x4**2 - 8*a**3*b*x2**3 + 8*a**3*b*x2**2*x3 + 8*a**3*b*x2**2*x4 + 8*a**3*b*x2*x3**2 - 16*a**3*b*x2*x3*x4 + 8*a**3*b*x2*x4**2 - 8*a**3*b*x3**3

## Grobner Bases for Algebraic Point Decomposition
This is Sage code.

In [ ]:
import random
# Field
F = GF(257)
# Curve: y^2 = x^3 + 2x + 2
a = F(2)
b = F(2)
E = EllipticCurve(F, [a, b])
print(E)

#Function to pick two small random points
def random_point_small_x(bound=40):
    while True:
        x = F(random.randint(0, bound))
        rhs = x^3 + a*x + b
        ys = [y for y in F if y^2 == rhs]
        if ys:
            return E(x, random.choice(ys))

P1 = random_point_small_x()
P2 = random_point_small_x()

# Find a target
T = P1 + P2
xT = F((T)[0])
print("P1 =", P1)
print("P2 =", P2)
print("T  =", T)

#Build factor base and include x(P1), x(P2)
FB = {F(P1[0]), F(P2[0])}
#Include random valid x-coordinates
while len(FB) < 10:
    x = F.random_element()
    rhs = x^3 + a*x + b
    if any(y^2 == rhs for y in F):
        FB.add(x)

FB = list(FB)
print("Factor base x-values:", FB)

#Define the polynomial ring alongside the third summation polynomial
R.<x1,x2> = PolynomialRing(F, order='lex')
def F3(x1, x2, x3):
    return ((x1 - x2)^2 * x3^2- 2*((x1 + x2)*(x1*x2 + a) + 2*b)*x3+ (x1*x2 - a)^2- 4*b*(x1 + x2))

#Construct the Grobner basis ideal
F_main = F3(x1, x2, xT)
fb_poly_1 = prod(x1 - xi for xi in FB)
fb_poly_2 = prod(x2 - xi for xi in FB)
I = Ideal([F_main, fb_poly_1, fb_poly_2])
G = I.groebner_basis()
print("Groebner basis:")
for g in G:
    print(g)

#Solve the system
sols = I.variety()
print("Solutions (x1, x2):", sols)

#Find corresponding y points
def lift_points(x):
    rhs = x^3 + a*x + b
    return [y for y in F if y^2 == rhs]

print("\nChecking solutions:")
for sol in sols:
    x1_sol = sol[x1]
    x2_sol = sol[x2]

    P1_candidates = [E(x1_sol, y) for y in lift_points(x1_sol)]
    P2_candidates = [E(x2_sol, y) for y in lift_points(x2_sol)]

    for Q1 in P1_candidates:
        for Q2 in P2_candidates:
            if Q1 + Q2 == T:
                print("Recovered decomposition:", Q1, "+", Q2, "=", T)